# Training on Google Colab
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/diego-dupuy/Clasificador-Neumonia-Pediatrica/blob/main/notebooks/train_colab.ipynb)

This notebook only **runs** the code in `src/`: it downloads the dataset, trains the three experiments,
evaluates them on the test set and generates the Grad-CAM figures.

**Before running:**
1. *Runtime → Change runtime type → T4 GPU*.
2. Kaggle key: create a token in kaggle.com → *Settings → API*, then in Colab click the 🔑 icon (left bar),
   add a secret called `KAGGLE_API_TOKEN` with the token as value and enable *Notebook access*.

In [ ]:
# 1) Download the code and install the only library missing in Colab
import os
if not os.path.exists("Clasificador-Neumonia-Pediatrica"):
    !git clone -q https://github.com/diego-dupuy/Clasificador-Neumonia-Pediatrica
os.chdir("Clasificador-Neumonia-Pediatrica")
!pip install -q grad-cam

In [ ]:
# 2) Download the dataset from Kaggle (the key is read from Colab secrets, never written in the code)
from google.colab import userdata
os.environ["KAGGLE_API_TOKEN"] = userdata.get("KAGGLE_API_TOKEN")
!kaggle datasets download -q -d paultimothymooney/chest-xray-pneumonia
!unzip -q -n chest-xray-pneumonia.zip -d data
!ls data/chest_xray

## What the data augmentation does
Each training epoch sees a slightly different version of every X-ray (random crop/zoom, small rotation,
brightness and contrast). Below: one training image and 5 random augmented versions.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from PIL import Image
from torchvision import transforms
from src.dataset import train_transform

path = sorted(os.listdir("data/chest_xray/train/PNEUMONIA"))[0]
image = Image.open(f"data/chest_xray/train/PNEUMONIA/{path}").convert("RGB")
augment = transforms.Compose(train_transform(True).transforms[:3])  # same steps, without normalising
fig, axes = plt.subplots(1, 6, figsize=(18, 3.5))
axes[0].imshow(image.resize((224, 224)), cmap="gray"); axes[0].set_title("original")
for ax in axes[1:]:
    ax.imshow(augment(image)); ax.set_title("augmented")
for ax in axes: ax.axis("off")
plt.show()

## Experiments
| Name | What changes |
|---|---|
| `baseline` | same set-up as my first version (but with a proper validation set) |
| `weighted` | + class weights in the loss, to compensate the imbalance (≈3 pneumonia per normal) |
| `weighted_aug` | + data augmentation |

Each one takes ~10-15 min on a T4.

In [ ]:
!python -m src.train --experiment baseline
!python -m src.evaluate --experiment baseline

In [ ]:
!python -m src.train --experiment weighted
!python -m src.evaluate --experiment weighted

In [ ]:
!python -m src.train --experiment weighted_aug
!python -m src.evaluate --experiment weighted_aug

## Comparison on the test set (624 X-rays)

In [ ]:
!python -m src.evaluate --compare

In [ ]:
from IPython.display import Image as Show, display
fig, axes = plt.subplots(1, 3, figsize=(15, 4.5))
for ax, name in zip(axes, ["baseline", "weighted", "weighted_aug"]):
    ax.imshow(plt.imread(f"results/{name}/confusion_matrix.png")); ax.axis("off")
plt.tight_layout(); plt.show()
fig, axes = plt.subplots(1, 3, figsize=(15, 3.8))
for ax, name in zip(axes, ["baseline", "weighted", "weighted_aug"]):
    ax.imshow(plt.imread(f"results/{name}/training_curves.png")); ax.axis("off")
plt.tight_layout(); plt.show()

## Grad-CAM: where does the model look?
`central_attention` = share of the heat map inside the central part of the image (15 % margin removed on
each side). If attention were spread evenly it would be ≈ 0.49; higher = more focused on the centre (lungs),
lower = more on borders and corners.

In [ ]:
!python -m src.gradcam --experiment baseline
!python -m src.gradcam --experiment weighted_aug

In [ ]:
for name in ["baseline", "weighted_aug"]:
    path = f"results/{name}/gradcam_false_negatives.png"
    if os.path.exists(path):
        display(Show(path))